In [9]:
import glob
import pandas as pd
import datetime
import os
from typing import List, Tuple, Any, Callable

from dorieh.cms.fts2yaml import mcr_type, MedicareFTS
from dorieh.platform.loader.data_loader import DataLoader
from dorieh.cms.mcr_data_loader import MedicareDataLoader
from dorieh.cms.tools.mcr_fts2db import MedicareLoader
from dorieh.utils.fwf import FWFReader
from dorieh.utils.io_utils import fopen


In [65]:
fts_path = "/n/dominici_nsaph_l3/Lab/data/ci3_d_medicare/original_data/cms_medicare/data/4334/2012/mbsf_ab_summary_res000017155_req004334_2012.fts"

f, ext = os.path.splitext(fts_path)
basedir, fname = os.path.split(f)
t = mcr_type(fname)
dat_path = f + ".dat"
parquet_path = f + ".parquet"

fts = MedicareFTS(t).init(fts_path)

def get_type(col):
    if col.type == 'CHAR':
        return 'str'
    elif col.type == "DATE":
        return "DATE"
    else:
        return float
    
# getting column names
fts_meta = fts.to_fwf_meta(dat_path)
colnm_lst = [col.name for col in fts_meta.columns]
colwidth_lst = [col.length for col in fts_meta.columns]
colspecs = [(col.start, col.end) for col in fts_meta.columns]
type_dict = {col.name: col.type for col in fts_meta.columns}
#type_lst = [col.type for col in fts_meta.columns]
#type_2 = 

# reading body of data frame
# with FWFReader(fts_meta) as reader:
#     # this gives us one row at a time
#     dat = []
#     record_count = 0 
#     for record in reader:
#         dat.append(record)
#         record_count += 1 
#         if record_count % 500000 == 0:
#             print(record_count)

# reading fixed width file via pandas
#df = pd.read_fwf(dat_path, widths=colwidth_lst, names=colnm_lst, index_col=False)

#df.head


In [8]:
df = pd.DataFrame(dat, columns=colnm_lst)

In [18]:
#df.head
col_info = pd.DataFrame({"colnm": colnm_lst,
                         "width": [col.length for col in fts_meta.columns],
                         "type": [col.type for col in fts_meta.columns]})
df.to_parquet("../data/temp/mbsf_ab_summary_res000017155_req004334_2012.parquet")

In [6]:
from itertools import zip_longest
from itertools import accumulate
import struct

def make_parser(fieldwidths):
    cuts = tuple(cut for cut in accumulate(abs(fw) for fw in fieldwidths))
    flds = tuple(zip((0,)+cuts, cuts))
    slcs = ', '.join(f'line[{i}:{j}]' for i, j in flds)
    parse = eval('lambda line: ({})\n'.format(slcs))  # Create and compile source code.
    # Optional informational function attributes.
    parse.size = sum(abs(fw) for fw in fieldwidths)
    parse.fmtstring = ' '.join('{}{}'.format(abs(fw), 'x' if fw < 0 else 's')
                                                for fw in fieldwidths)
    return parse

#fieldwidths = (2, -10, 24)  # negative widths represent ignored padding fields
parse = make_parser(colwidth_lst)
#fields = parse(line)
print('format: {!r}, rec size: {} chars'.format(parse.fmtstring, parse.size))
#print('fields: {}'.format(fields))

format: '15s 4s 1s 1s 8s 2s 2s 3s 9s 3s 8s 1s 8s 8s 1s 1s 1s 1s 1s 1s 2s 1s 1s 3s 3s 3s 3s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 1s 2s 4s 9s', rec size: 134 chars


In [7]:
# read all info
with open(dat_path, "r") as f:
    lines = f.read().split("\n")

In [52]:
tmp = pd.DataFrame([parse(line) for line in lines[:10000000]], columns=colnm_lst)

In [59]:
tmp2 = tmp.copy()
# from dateutil.parser import parse as date_parser
# t2 = [date_parser(entry.strip()) for entry in tmp["BENE_DOB"]]


In [66]:
t0 = datetime.datetime.now()
# dt_format = pd.to_datetime(tmp['COVSTART'], errors='coerce')
# t1 = datetime.datetime.now() - t0
# print(t1)

for col in tmp2.columns:
    print(col)
    if type_dict[col] == 'NUM':
        tmp2[col] = pd.to_numeric(tmp2[col].str.strip(), errors='coerce')
    elif type_dict[col] == 'DATE':
        tmp2[col] = pd.to_datetime(tmp2['COVSTART'], errors='coerce')

t1 = datetime.datetime.now() - t0
print(t1)

BENE_ID
RFRNC_YR
FIVEPCT
EFIVEPCT
COVSTART
CRNT_BIC
STATE_CD
CNTY_CD
BENE_ZIP
AGE
BENE_DOB
V_DOD_SW
DEATH_DT
NDI_DEATH_DT
SEX
RACE
RTI_RACE_CD
OREC
CREC
ESRD_IND
MS_CD
A_TRM_CD
B_TRM_CD
A_MO_CNT
B_MO_CNT
BUYIN_MO
HMO_MO
BUYIN01
BUYIN02
BUYIN03
BUYIN04
BUYIN05
BUYIN06
BUYIN07
BUYIN08
BUYIN09
BUYIN10
BUYIN11
BUYIN12
HMOIND01
HMOIND02
HMOIND03
HMOIND04
HMOIND05
HMOIND06
HMOIND07
HMOIND08
HMOIND09
HMOIND10
HMOIND11
HMOIND12
STATE
YEAR
ZIP
0:01:03.029880


In [64]:
type_dict

{'BENE_ID': {'type': 'CHAR', 'digits': None},
 'RFRNC_YR': {'type': 'NUM', 'digits': None},
 'FIVEPCT': {'type': 'CHAR', 'digits': None},
 'EFIVEPCT': {'type': 'CHAR', 'digits': None},
 'COVSTART': {'type': 'DATE', 'digits': None},
 'CRNT_BIC': {'type': 'CHAR', 'digits': None},
 'STATE_CD': {'type': 'CHAR', 'digits': None},
 'CNTY_CD': {'type': 'CHAR', 'digits': None},
 'BENE_ZIP': {'type': 'CHAR', 'digits': None},
 'AGE': {'type': 'NUM', 'digits': None},
 'BENE_DOB': {'type': 'DATE', 'digits': None},
 'V_DOD_SW': {'type': 'CHAR', 'digits': None},
 'DEATH_DT': {'type': 'DATE', 'digits': None},
 'NDI_DEATH_DT': {'type': 'DATE', 'digits': None},
 'SEX': {'type': 'CHAR', 'digits': None},
 'RACE': {'type': 'CHAR', 'digits': None},
 'RTI_RACE_CD': {'type': 'CHAR', 'digits': None},
 'OREC': {'type': 'CHAR', 'digits': None},
 'CREC': {'type': 'CHAR', 'digits': None},
 'ESRD_IND': {'type': 'CHAR', 'digits': None},
 'MS_CD': {'type': 'CHAR', 'digits': None},
 'A_TRM_CD': {'type': 'CHAR', 'digit